# Data analyses
1. Regression analysis
2. Confounding analysis
3. Mediation analysis
4. PSM analysis
5. CEM analysis

All the above analyses use the same dataframe ("data/processed/PaperID_KI2-Dopen_nok_control.pickle") merged from the processed data. You can process it by yourself via the code in "notebooks/0_data_preparing.ipynb".

## 1. Normalized ordinary-least-squares (OLS) regression
To analyze the relationships between scientific disruption and various covariates, we employ normalized ordinary least squares (OLS) regression models. This approach allows us to evaluate whether $\rm{KI}$ has a dominant effect on disruption. The regression models include the following covariates: focal paper properties (impact $C_5$, discipline, and publication year), reference properties (novelty, multidisciplinarity, reference count, average age, average impact $C_5$, and average disruption), as well as team properties (size, geographic distance, and collaboration freshness). All numerical variables are standardized by $Z$-score normalization to ensure comparability of effect coefficients across models. We start with a model that does not control for any covariates other than fixing discipline and publication year (eq.2). Then, we control for each of the following covariates separately (eq.3): focal paper's impact ($C_5$), novelty, multidisciplinarity, reference count, reference age, reference impact ($C_5$), reference disruption, team size, team distance, and team freshness. Finally, we consider a model that accounts for all those covariates (eq.4).

$$
\begin{aligned}
    D &= b_0 + a_0{\rm{KI}} + c_{0}F, \quad\quad\quad\quad\quad\quad\quad\quad (2) \\
    D &= b_0 + a_0{\rm{KI}} + a_{i}C_i + c_{0}F, \quad\quad\quad\quad\quad (3) \\
    D &= b_0 + a_0{\rm{KI}} + \sum\nolimits_{i=1}^{10}a_{i}C_i + c_{0}F. \quad\quad (4)
\end{aligned}
$$

In [ ]:
import sys
import os
from scipy import stats
import statsmodels.api as sm
import pandas as pd
import numpy as np
from dominance_analysis import Dominance # modified for category variables
from collections import defaultdict

pre_path = os.path.abspath(r"..")
sys.path.insert(1, os.path.join(pre_path, 'src'))
from utils import load_data, dataframe_nor, rank, rank_bin

In [ ]:
# Set parameters
data_type = 'nor'      # 'raw' or 'nor': normalized or raw data
DC_type = 'Dopen_nok'  # 'Dopen_nok', 'Dopen', 'D5_nok', 'D5'
KI_type = 'KI2'        # 'KI2', 'KI2_frac', 'KI2_adj', 'KI2_adj_frac'
file_path = '%s/data/processed/PaperID_%s-%s_merged.pickle'%(pre_path,KI_type,DC_type)

# Define dependent and independent variables, fixed effects, and control variables
dependent_var = DC_type
independent_var = KI_type
fixed_effects = ['Year','Field']  # fixed effects variables that'll be converted to dummy (categorical) variables

# control all covirate variables, one can add or remove variables as needed
control_vars = ['log_C5','Multidiscipline','Novelty_90pct','reference_Count','reference_Age','reference_C5','reference_'+DC_type,'Team_Size','Team_Distance','Team_Freshness']


# Define all variables to be used in the model
all_vars = [dependent_var,independent_var] + fixed_effects + control_vars
# load data
df = load_data(file_path, data_type, all_vars, fixed_effects)

# Define the model
df_processed = pd.get_dummies(df, columns=fixed_effects, drop_first=True, dtype=int) # drop_first=True avoid multicollinearity issues, it generates k-1 dummy variables for k categories
del df  # free memory

# Prepare dependent and independent variables
y = df_processed[dependent_var]
X = df_processed.drop(dependent_var, axis=1)
X = sm.add_constant(X) # add constant term for the intercept

model = sm.OLS(y, X)    # Describe model
res = model.fit()       # Fit model
print(res.summary())    # Summarize model
# Save the results
with open('%s/results/results_for_tables/Regression_Analysis_Results/Regression_summary_%s_%s_%s.latex'%(pre_path,KI_type,DC_type,data_type), 'w') as fh:
    fh.write(res.summary().as_latex())

### Dominance Analysis
To further quantify the relative importance of individual predictors within the full model and to systematically decompose the explained variance ($R^2$), we performed a dominance analysis. This method systematically assesses each predictor's marginal contribution to $R^2$ across all possible sub-models, thereby attributing its unique and shared contributions to the total $R^2$. It establishes three types of dominance: conditional dominance (contribution in specific subsets), partial dominance (average contribution across specific subset sizes), and general dominance (overall average contribution across all subsets). The general dominance value, which inherently accounts for shared variance and avoids order dependence, serves as our primary measure of relative importance for each predictor. This approach allows for a robust and comprehensive comparison of the relative influence of all independent variables and fixed effects in the model.

In [ ]:
from scipy import stats
import pandas as pd
import numpy as np
from itertools import combinations
import math
from sklearn.linear_model import LinearRegression
from collections import defaultdict

def canonical_key(subset_tuple):
    """return a tuple sorted according to the order in `conceptual_predictors` as cache key to ensure consistency."""
    if not subset_tuple:
        return tuple()
    return tuple(sorted(subset_tuple, key=lambda x: concept_order[x]))

def get_r2(X, y, predictor_indices):
    """a helper function to fit model and return R-squared"""
    if not predictor_indices:
        return 0.0  # no predictor model, R-squared is 0
    X_sub = X[:, predictor_indices]
    # to ensure X_sub is 2D for sklearn
    if X_sub.ndim == 1:
        X_sub = X_sub.reshape(-1, 1)
        
    # linear regression using sklearn, which is faster than statsmodels for large data
    model = LinearRegression().fit(X_sub, y)  # default： fit_intercept=True，yielding same results as statsmodels with constant
    R2 = model.score(X_sub, y)

    '''X_sub = sm.add_constant(X_sub, has_constant='add')
    model = sm.OLS(y, X_sub).fit()
    R2 = model.rsquared'''

    return R2

def dominance_analysis(conceptual_predictors, predictor_map_idx, X_np, y_np):
    """
    this function calculates the general dominance of each conceptual predictor by:
    1. calculate and cache R-squared for all possible sub-models.
    2. traverse all subsets of other predictors (including empty set)
    3. compute marginal contribution of P in context C as:
    4. taking the average of marginal contributions across all contexts as P's general dominance. 
    """
    print("\n--- start the dominance analysis ---")
    r2_cache = {}
    
    # 1: calculate all models' R-squared and cache them (suggest to use parallel computing for speedup if needed)
    for i in range(len(conceptual_predictors) + 1):
        for subset in combinations(conceptual_predictors, i):
            # notice：combinations keeps the order of input tuple, so we can directly use tuple(subset) as key
            actual_indices = [idx for concept in subset for idx in predictor_map_idx[concept]]
            cache_key = canonical_key(tuple(subset))
            r2_cache[cache_key] = get_r2(X_np, y_np, actual_indices)

    print(f"Completed {len(r2_cache)} models' R-squared.")

    # 2 & 3 & 4: calculate the general dominance of each variable
    general_dominance = defaultdict(float)
    contributions = defaultdict(list)

    for predictor in conceptual_predictors:
        # get the other predictors
        others = [p for p in conceptual_predictors if p != predictor]
        
        # traversing all subsets of other predictors (including empty set)
        for i in range(len(others) + 1):
            for subset in combinations(others, i):
                # key keeps the order
                model_without_p = canonical_key(tuple(subset))
                model_with_p = canonical_key(tuple(subset + (predictor,)))
                # search from cache and compute marginal contribution
                marginal_contribution = r2_cache[model_with_p] - r2_cache[model_without_p]
                subset_size = len(subset)
                contributions[predictor].append((subset_size, marginal_contribution))
    
    # calculate the average contributions, i.e., general dominance
    # applying Shapley weights to ensure additivity
    k = len(conceptual_predictors)
    fact_k = math.factorial(k)
    for predictor, contrib_list in contributions.items():
        # contrib_list stores tuples (subset_size, marginal_contribution)
        total = 0.0
        for subset_size, marginal_contribution in contrib_list:
            weight = math.factorial(subset_size) * math.factorial(k - subset_size - 1) / fact_k
            total += weight * marginal_contribution
        general_dominance[predictor] = total

    return general_dominance


#1. Prepare groups for dominance analysis
conceptual_predictors = [independent_var]+control_vars+fixed_effects

#2. Group dummy variables by their original categorical variable
predictor_map = {}
for var in conceptual_predictors:
    if var not in fixed_effects:
        # continuous variables: map to itself
        predictor_map[var] = [var]
    else:
        # fixed effects (categorical variables): map to all dummy variables
        predictor_map[var] = [col for col in df_processed.columns if col.startswith(f'{var}_')]
print("\n--- mapping conceptual predictors to actual columns ---")
print(predictor_map)

#3. Prepare data for dominance analysis
feature_columns = [col for col in df_processed.columns if col != dependent_var]
y_np = df_processed[dependent_var].values.astype(np.float32)
X_np = df_processed[feature_columns].values.astype(np.float32)
del df_processed  # release memory

#4. predictor_map_idx: conceptual -> corresponding column indices in feature_columns
predictor_map_idx = {}
for concept, cols in predictor_map.items():
    predictor_map_idx[concept] = [feature_columns.index(c) for c in cols]

#5. To maintain consistency in dictionary keys, create indices in the order of conceptual_predictors
concept_order = {c: i for i, c in enumerate(conceptual_predictors)}

#6. run analysis
general_dominance_stats = dominance_analysis(
    conceptual_predictors=conceptual_predictors,
    predictor_map_idx=predictor_map_idx,
    X_np=X_np,
    y_np=y_np
)

dominance_stats = pd.DataFrame.from_dict(general_dominance_stats, orient='index', columns=['General Dominance'])
dominance_stats = dominance_stats.sort_values(by='General Dominance', ascending=False)
sum_of_dominance = dominance_stats['General Dominance'].sum()
dominance_stats['Relative Importance (%)'] = (dominance_stats['General Dominance'] / sum_of_dominance) * 100
print(dominance_stats)
print('\n\n\n\n')

# Save the results
dominance_stats.to_csv('%s/results/results_for_tables/Regression_Analysis_Results/Dominance_summary_%s_%s_%s.latex'%(pre_path,KI_type,DC_type,data_type), index=True)

## 2. Confounding analysis
Confounding is a critical challenge in causal inference, where an a priori variable $C$ can distort the true effect of a target exposure $X$ on an outcome $Y$ ($C \rightarrow X, C \rightarrow Y$), leading to spurious associations or biased estimates ($X \dashrightarrow Y$).
This analysis aims to rigorously estimate the causal effect of a target variable $X$ on an outcome $Y$ by identifying and adjusting for potential confounding factors that are common causes of both $X$ and $Y$, and are not causally affected by $X$. Here, we employ a model-based confounding analysis framework, comparing two parametric regression models. The first model, the Unadjusted Model (eq.5), formulates the baseline association between the outcome $Y$ and the target variable $X$, while controlling for other relevant covariates $F$. The coefficient $\alpha_1$ in this model represents the unadjusted (or crude) effect of $X$ on $Y$. The second model, the Adjusted Model (eq.6), expands upon the first by explicitly incorporating the identified confounder $C$ alongside $X$ and other covariates $F$. The coefficient $\delta_1$ in this model captures the adjusted effect of $X$ on $Y$, after accounting for the confounding effect of $C$. The presence and magnitude of confounding are primarily assessed by observing the change in the estimated coefficient of $X$ when the confounder $C$ is introduced into the model. The relative change in estimate (RCE) is defined as $|\frac{\alpha_1-\delta_1}{\alpha_1}|$. A substantial RCE indicates that $C$ indeed played a confounding role, distorting the unadjusted association between $X$ and $Y$. This adjustment helps to isolate $X$'s independent contribution to $Y$. 
In all conducted confounding analyses, we standardize the numerical variables by Z-score normalization to ensure comparability of effect coefficients across models, with discipline and publication year controlled.

$$
\begin{aligned}
Y &= \alpha_{0} + \alpha_{1}X + \alpha_{2}F, \quad\quad\quad\quad\quad\quad\quad\quad (5)\\
Y &= \delta_{0} + \delta_{1}X + \delta_{2}C + \delta_{3}F. \quad\quad\quad\quad\quad\quad (6)
\end{aligned}
$$

In [ ]:
def P_value_stars(pv):
    stars = ''
    if  pv< 0.05 and pv>= 0.01:
        # stars = '*'
        stars = 1
    if  pv< 0.01 and pv>= 0.001:
        # stars = '**'
        stars = 2
    if  pv< 0.001:
        # stars = '***'
        stars = 3
    return stars
    
    
def summary_sub(effect_list, smry, confounder, effect_name):
    smry.loc[confounder, effect_name] = np.median(effect_list) # Estimate
    
    Statistic_T, P_value = stats.ttest_1samp(effect_list, 0) # Statistic T, P-value
    smry.loc[confounder, effect_name+"_PV"] = P_value # P-value
    smry.loc[confounder, effect_name+"_Stars"] = P_value_stars(P_value) # Significance stars
    
    Lower_CI_bound, Upper_CI_bound = np.percentile(effect_list, [2.5, 97.5]) # Lower CI bound, Upper CI bound
    smry.loc[confounder, effect_name+"_SE"] = (Upper_CI_bound-Lower_CI_bound)/2 # SE
    
    return smry


def summary(Confounder_list, unadjusted_effects, adjusted_effects):
    """
    Provide a summary of a confounding analysis.
    """

    columns = ["UE", "UE_PV", "UE_Stars", "UE_SE", "AE", "AE_PV", "AE_Stars", "AE_SE"]
    smry = pd.DataFrame(columns=columns, index=Confounder_list)
    smry.index.name = "Confounder"
    
    for confounder in Confounder_list:
        smry = summary_sub(unadjusted_effects[confounder], smry, confounder, "UE")
        smry = summary_sub(adjusted_effects[confounder], smry, confounder, "AE")
    smry = smry.apply(pd.to_numeric, errors='coerce')
    smry["RCE"] = abs(smry["UE"]-smry["AE"]) / abs(smry["UE"])
    # smry = smry.reset_index()
    print(smry)
    return smry

In [ ]:
# Set parameters
data_type = 'raw'    # 'raw' or 'nor'
n_bootstraps = 1000      # Number of iterations for bootstrapping

DC_type = 'Dopen_nok'  # 'Dopen_nok', 'Dopen', 'D5_nok', 'D5'
KI_type = 'KI2'        # 'KI2', 'KI2_frac', 'KI2_adj', 'KI2_adj_frac'
file_path = '%s/data/processed/PaperID_%s-%s_merged.pickle'%(pre_path,KI_type,DC_type)

# Define dependent and independent variables, fixed effects, and control variables
dependent_var = DC_type
independent_vars = ['reference_DC'+DC_type,'Citation_percentile']
Confounding_vars = [KI_type,'Novelty_90pct','Multidiscipline','reference_Count','reference_Age','reference_C5','Team_Size','Team_Distance','Team_Freshness']#
fixed_effects = ['Year','Field'] # fixed effects variables that'll be converted to dummy (categorical) variables
###################
for independent_var in independent_vars: # Due to the time-consuming nature of the analysis, we suggest to analyze the variable one by one.
    # 1. Filter confounders
    Confounder_list = [C for C in Confounding_vars if C != independent_var] # Exclude the independent variable from the Confounder list
    
    # 2. load data
    all_vars = [dependent_var,independent_var] + Confounder_list + fixed_effects
    df = load_data(file_path, data_type, all_vars, fixed_effects) # data_type sets 'raw' first to do within transformation, then 'nor' to do normalization

    # 3. Within Transformation (Two-Way Fixed Effects Demeaning)
    vars_to_process = [dependent_var, independent_var] + Confounder_list
    df_demeaned = pd.DataFrame(index=df.index)   # initialize an empty DataFrame, keep the original index
    for col in vars_to_process:
        # explicitly pass observed=True to keep current behavior with categorical dtypes
        field_mean = df.groupby('Field', observed=True)[col].transform('mean')
        year_mean = df.groupby('Year', observed=True)[col].transform('mean')
        grand_mean = df[col].mean()
        df_demeaned[col] = df[col] - field_mean - year_mean + grand_mean
    del df

    # 4. standardizing dataframe for comparability between variables
    df_demeaned = dataframe_nor(df_demeaned)

    # 5. Prepare data for bootstrapping
    y_o = df_demeaned[dependent_var].values.astype(np.float32)
    X_c_all = df_demeaned[[independent_var] + Confounder_list].values.astype(np.float32) # exclude independent_var and fixed effects
    df_len = len(y_o)
    del df_demeaned

    # 6. bootstrapping
    unadjusted_effects = defaultdict(list)
    adjusted_effects = defaultdict(list)
    for iter in range(n_bootstraps): # (suggest to use parallel computing for speedup if needed)
        boot_idx = np.random.randint(df_len, size=df_len)

        # bootstrap samples
        y_o_boot = np.take(y_o, boot_idx, axis=0)
        X_c_all_boot = np.take(X_c_all, boot_idx, axis=0)
        X_o_boot = X_c_all_boot[:, [0]]  # shape (n,2) independent_var as X
        
        # fit the basic unadjusted model
        model_simple = sm.OLS(y_o_boot, X_o_boot).fit()    # shape (n,1), [independent_var]
        # calculate unadjusted effect
        alpha_unadjusted = model_simple.params[0]

        # for each possible Confounder, fit the adjusted models, and calculate effects
        for conf_idx, Confounder in enumerate(Confounder_list):
            confounder_col = conf_idx + 1 # because the first column is independent_var

            # bootstrap samples
            X_c_boot = X_c_all_boot[:, [0, confounder_col]] # shape (n,2) [independent_var, Confounder]

            # fit the adjusted model, without constant
            model_adjusted = sm.OLS(y_o_boot, X_c_boot).fit()  # shape (n,1), [independent_var, Confounder]
            # calculate adjusted effect
            delta_adjusted = model_adjusted.params[0]
            
            unadjusted_effects[Confounder].append(alpha_unadjusted)
            adjusted_effects[Confounder].append(delta_adjusted)
    summary(Confounder_list, unadjusted_effects, adjusted_effects).to_csv('%s/results/results_for_tables/Confounding_Analysis_Results/Confounding_Analysis_%s_%s_%s_%s.csv'%(pre_path,KI_type,DC_type,data_type,independent_var),encoding='utf-8')


## 3. Mediation analysis
Mediation analysis is often applied to data from randomized controlled trials, aiming to investigate whether and how the effect of a target variable $X$ on an outcome $Y$ operates through a mediating variable $M$ ($X\rightarrow M \rightarrow Y$), where $M$ is a post-hoc variable relative to $X$ and an a priori variable relative to $Y$.
Here we utilize the model-based mediation framework, comprising two parametric regression models: Outcome Model (eq.7) formulates the relation between the outcome $Y$ and the target variable $X$, as well as the mediator $M$, with covariates $C$ controlled. On the other hand, the Mediator Model (eq.8) formulates the relation between the mediator $M$ and the target variable $X$, also controlling for covariates $C$. The indirect effect (ACME) is defined as the product of the coefficient $\beta_1$ of $X$ in model (eq.8) and the coefficient $\theta_2$ of $M$ in model (eq.7). This represents the portion of $X$'s effect on $Y$ mediated through $M$. The direct effect (ADE) is defined as the coefficient $\theta_1$ of $X$ in model (eq.7), capturing the direct relationship between $X$ and $Y$, independent of $M$. The total effect (TE) is defined as the sum of the indirect effect (ACME) and direct effect (ADE). 
In all conducted mediation analyses (i.e., team compositions as the target variables), we standardize the numerical variables by $Z$-score normalization to ensure comparability of effect coefficients across models, with discipline and publication year controlled. 
$$
\begin{aligned}
    Y &= \theta_{0} + \theta_{1}X + \theta_{2}M + \theta_{3}C, \quad (7)\\
    M &= \beta_{0} + \beta_{1}X + \beta_{2}C. \quad\quad\quad\quad (8)
\end{aligned}
$$

In [ ]:
def P_value_stars(pv):
    stars = ''
    if  pv< 0.05 and pv>= 0.01:
        # stars = '*'
        stars = 1
    if  pv< 0.01 and pv>= 0.001:
        # stars = '**'
        stars = 2
    if  pv< 0.001:
        # stars = '***'
        stars = 3
    return stars
    
    
def summary_sub(effect_list, smry, mediator, effect_name):
    smry.loc[mediator, effect_name] = np.median(effect_list) # Estimate
    
    Statistic_T, P_value = stats.ttest_1samp(effect_list, 0) # Statistic T, P-value
    smry.loc[mediator, effect_name+"_PV"] = P_value # P-value
    smry.loc[mediator, effect_name+"_Stars"] = P_value_stars(P_value) # Significance stars
    
    Lower_CI_bound, Upper_CI_bound = np.percentile(effect_list, [2.5, 97.5]) # Lower CI bound, Upper CI bound
    smry.loc[mediator, effect_name+"_SE"] = (Upper_CI_bound-Lower_CI_bound)/2 # SE
    
    return smry


def summary(Mediator_list, indirect_effects, direct_effects, total_effects):
    """
    Provide a summary of a mediation analysis.
    """

    columns = ["ACME", "ACME_PV", "ACME_Stars", "ACME_SE", "ADE", "ADE_PV", "ADE_Stars", "ADE_SE", "TE", "TE_PV", "TE_Stars", "TE_SE"]
    smry = pd.DataFrame(columns=columns, index=Mediator_list)
    smry.index.name = "Mediator"
    
    for mediator in Mediator_list:
        smry = summary_sub(indirect_effects[mediator], smry, mediator, "ACME")
        smry = summary_sub(direct_effects[mediator], smry, mediator, "ADE")
        smry = summary_sub(total_effects[mediator], smry, mediator, "TE")
    smry = smry.apply(pd.to_numeric, errors='coerce')
    smry["ACME/TE"] = abs(smry["ACME"]) / abs(smry["TE"])
    # smry = smry.reset_index()
    print(smry)
    return smry

In [ ]:
# Set parameters
data_type = 'raw'    # 'raw' or 'nor'
n_bootstraps = 1000      # Number of iterations for bootstrapping

DC_type = 'Dopen_nok'  # 'Dopen_nok', 'Dopen', 'D5_nok', 'D5'
KI_type = 'KI2'        # 'KI2', 'KI2_frac', 'KI2_adj', 'KI2_adj_frac'
file_path = '%s/data/processed/PaperID_%s-%s_merged.pickle'%(pre_path,KI_type,DC_type)

# Define dependent and independent variables, fixed effects, and control variables
dependent_var = DC_type
independent_vars = ['Team_Size','Team_Distance','Team_Freshness']
Mediating_vars = [KI_type,'Novelty_90pct','Multidiscipline','reference_Count','reference_Age','reference_C5','Team_Size','Team_Distance','Team_Freshness']#
fixed_effects = ['Year','Field'] # fixed effects variables that'll be converted to dummy (categorical) variables
###################
for independent_var in independent_vars: # Due to the time-consuming nature of the analysis, we suggest to analyze the variable one by one.
    # 1. Filter mediators
    Mediator_list = [M for M in Mediating_vars if M != independent_var] # Exclude the independent variable from the mediators list
    
    # 2. load data
    all_vars = [dependent_var,independent_var] + Mediator_list + fixed_effects
    df = load_data(file_path, data_type, all_vars, fixed_effects) # data_type sets 'raw' first to do within transformation, then 'nor' to do normalization

    # 3. Within Transformation (Two-Way Fixed Effects Demeaning)
    vars_to_process = [dependent_var, independent_var] + Mediator_list
    df_demeaned = pd.DataFrame(index=df.index)   # initialize an empty DataFrame, keep the original index
    for col in vars_to_process:
        # explicitly pass observed=True to keep current behavior with categorical dtypes
        field_mean = df.groupby('Field', observed=True)[col].transform('mean')
        year_mean = df.groupby('Year', observed=True)[col].transform('mean')
        grand_mean = df[col].mean()
        df_demeaned[col] = df[col] - field_mean - year_mean + grand_mean
    del df

    # 4. standardizing dataframe for comparability between variables
    df_demeaned = dataframe_nor(df_demeaned)

    # 5. Prepare data for bootstrapping
    y_o = df_demeaned[dependent_var].values.astype(np.float32)
    X_o_all = df_demeaned[[independent_var] + Mediator_list].values.astype(np.float32) # columns for outcome model

    df_len = len(y_o)
    del df_demeaned

    # 6. bootstrapping
    indirect_effects = defaultdict(list)
    direct_effects = defaultdict(list)
    total_effects = defaultdict(list)
    for iter in range(n_bootstraps): # (suggest to use parallel computing for speedup if needed)
        boot_idx = np.random.randint(df_len, size=df_len)

        # bootstrap samples
        y_o_boot = np.take(y_o, boot_idx, axis=0)
        X_o_all_boot = np.take(X_o_all, boot_idx, axis=0)
        
        # for each possible mediator, fit the outcome and mediator models, and calculate effects
        for conf_idx, Mediator in enumerate(Mediator_list):
            mediator_col = conf_idx + 1 # because the first column is independent_var

            # bootstrap samples
            X_o_boot = X_o_all_boot[:, [0, mediator_col]] # shape (n,2) [independent_var, Mediator]
            y_m_boot = X_o_boot[:, 1]                     # shape (n,1) Mediator as Y
            X_m_boot = X_o_boot[:, [0]]                   # shape (n,2) independent_var as X

            # fit the outcome model
            reg = sm.OLS(y_o_boot, X_o_boot).fit() # automatically handles categorical variables

            # fit the mediator model
            med = sm.OLS(y_m_boot, X_m_boot).fit() # automatically handles categorical variables

            # calculate the direct, indirect, and total effects
            direct = reg.params[0]
            indirect = med.params[0] * reg.params[1]
            total = direct + indirect
            
            indirect_effects[Mediator].append(indirect)
            direct_effects[Mediator].append(direct)
            total_effects[Mediator].append(total)
    summary(Mediator_list, indirect_effects, direct_effects, total_effects).to_csv('%s/results/results_for_tables/Mediation_Analysis_Results/Mediation_Analysis_%s_%s_%s_%s.csv'%(pre_path,KI_type,DC_type,data_type,independent_var),encoding='utf-8')


## 4. Propensity Score Matching (PSM) analysis
The procedure involves the following steps. We first categorize papers into the treated group and controlled group according to treatment size ($\rm{KI}$). Subsequently, we estimate the propensity score for each paper by fitting a generalized linear model where the treatment size is the dependent variable. The model includes the following covariates: focal paper properties (impact $C_5$, discipline, and publication year), reference properties (novelty, multidisciplinarity, reference count, average age, average impact $C_5$, and average disruption), and team properties (size, geographic distance, and collaboration freshness). For each paper $i$ in the controlled group, a counterpart from the treated group with a similar propensity score is matched. This matching ensures a balanced distribution of covariates across the two groups, effectively mimicking the conditions of a randomized controlled trial. After identifying $m$ matched pairs of papers, denoted as $M$, we calculate the Average Treatment Effect on the Treated as $\text{ATT}=\frac{1}{m}\sum_{i\in M}(Y^{1}_{i}-Y^{0}_{i})$, where $Y^{1}_{i}$ is the outcome (scientific disruption) for the treated paper and $Y^{0}_{i}$ is the outcome for the control paper. To enhance the robustness of PSM results, we perform multiple experiments for different pairs of treated and control groups. This will produce a matrix of ATT values, providing a comprehensive and nuanced view of the effect size of $\rm{KI}$ on scientific disruption.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from concurrent.futures import ProcessPoolExecutor, as_completed


# --- Causal Inference Method Functions ---
def preprocess_features_once_PSM(df_temp, Treatment, Outcome, non_numeric_cols):

    # 1. One-hot encoding
    if non_numeric_cols:
        df_encoded = pd.get_dummies(df_temp, columns=non_numeric_cols, drop_first=True)
        feature_cols = [col for col in df_encoded.columns 
                       if col not in [Treatment, Outcome]]
    else:
        df_encoded = df_temp.copy()
        feature_cols = [col for col in df_encoded.columns 
                       if col not in [Treatment, Outcome]]
    
    # 2. Data type optimization - choose the smallest data type based on the data range
    X_features = df_encoded[feature_cols].values.astype(np.float32) # Use float32 to save memory
    y_treatment = df_encoded[Treatment].values
    y_outcome = df_encoded[Outcome].values.astype(np.float32)

    # 3. Memory-aligned contiguous arrays (key optimization)
    n_samples, n_features = X_features.shape

    # Use structured arrays for optimal memory locality
    dtype_list = [('features', np.float32, (n_features,)), 
                  ('treatment', y_treatment.dtype), 
                  ('outcome', np.float32)]
    
    structured_array = np.empty(n_samples, dtype=dtype_list)
    structured_array['features'] = X_features
    structured_array['treatment'] = y_treatment
    structured_array['outcome'] = y_outcome
    
    preprocessing_info = {
        'structured_data': structured_array,
        'n_features': n_features,
        'feature_names': feature_cols,
        'n_samples': n_samples
    }
    
    return preprocessing_info


def psm_batch_process(args):
    """batch processing"""
    preprocessing_info, control_size, treatment_size, treat_effect_type, seed_batch = args
    
    try:
        results = []
        n_samples = preprocessing_info['n_samples']
        
        for seed in seed_batch:
            try:    
                np.random.seed(seed)
                bootstrap_indices = np.random.randint(0, n_samples, size=n_samples)

                # use the same bootstrapped data structure
                bootstrap_data = preprocessing_info['structured_data'][bootstrap_indices]
                
                X_bootstrap = bootstrap_data['features']
                y_treatment_bootstrap = bootstrap_data['treatment']
                y_outcome_bootstrap = bootstrap_data['outcome']

                treatment_size_typed = np.array(treatment_size, dtype=y_treatment_bootstrap.dtype)
                treatment_binary = (y_treatment_bootstrap == treatment_size_typed)
                
                if not (treatment_binary.any() and (~treatment_binary).any()):
                    continue

                # logic regression
                lr = LogisticRegression(solver='liblinear', max_iter=1000)
                lr.fit(X_bootstrap, treatment_binary)
                propensity_scores = lr.predict_proba(X_bootstrap)[:, 1]
                
                treatment_mask = treatment_binary
                control_mask = ~treatment_binary
                
                control_scores = propensity_scores[control_mask]
                treatment_scores = propensity_scores[treatment_mask]
                
                nn = NearestNeighbors(n_neighbors=1, algorithm='auto', p=2)
                nn.fit(control_scores.reshape(-1, 1))
                distances, indices = nn.kneighbors(treatment_scores.reshape(-1, 1))
                
                matched_control_outcomes = y_outcome_bootstrap[control_mask][indices.flatten()]
                treatment_outcomes = y_outcome_bootstrap[treatment_mask]
                
                att = np.mean(treatment_outcomes) - np.mean(matched_control_outcomes)
                
                if np.isfinite(att):
                    results.append(float(att))
                    
            except Exception as e:
                print(f"Error processing seed {seed}: {str(e)[:50]}")
                continue
        
        return results
        
    except Exception as e:
        print(f"Error in batch processing: {str(e)[:100]}")
        return []


def PSM_analysis_bootstrap(df_temp, Treatment, Outcome, non_numeric_cols, control_size, treatment_size, treat_effect_type, n_bootstraps, CPU_COUNT):    
    # preprocessing
    preprocessing_info = preprocess_features_once_PSM(df_temp, Treatment, Outcome, non_numeric_cols)
    
    # batch processing strategy - each process handles multiple bootstraps
    batch_size = max(1, n_bootstraps // (CPU_COUNT * 2))
    seed_batches = []
    
    for i in range(0, n_bootstraps, batch_size):
        batch_seeds = list(range(42 + i, 42 + min(i + batch_size, n_bootstraps))) # 42 is the random seed to ensure reproducibility
        seed_batches.append(batch_seeds)
    
    args_list = [(preprocessing_info, control_size, treatment_size, treat_effect_type, batch_seeds) 
                 for batch_seeds in seed_batches]
    
    att_bootstraps = []
    
    # set less processes but increase workload for each process
    optimal_workers = min(CPU_COUNT, len(seed_batches))
    
    with ProcessPoolExecutor(max_workers=optimal_workers) as executor:
        futures = [executor.submit(psm_batch_process, args) for args in args_list]
        for i, future in enumerate(as_completed(futures)):
            if i % max(1, len(futures) // 10) == 0:
                print(f"PSM: Completed batch {i+1}/{len(futures)}")
            
            batch_results = future.result()
            att_bootstraps.extend(batch_results)

    # quick return if no valid results
    if not att_bootstraps:
        return np.nan, np.nan, np.nan, np.nan
    
    att_array = np.array(att_bootstraps, dtype=np.float32)
    
    if len(att_array) < 2:
        return float(np.mean(att_array)), np.nan, np.nan, np.nan
    
    # vectorized statistics
    overall_att = float(np.mean(att_array))
    ci_lower = float(np.percentile(att_array, 2.5))
    ci_upper = float(np.percentile(att_array, 97.5))
    
    # calculate p-value
    positive_prop = np.mean(att_array > 0)
    p_value = float(2 * min(positive_prop, 1 - positive_prop))
    
    return overall_att, p_value, ci_lower, ci_upper

In [ ]:
# Set parameters
n_bootstraps = 1000      # Number of iterations for bootstrapping
CPU_COUNT = 36           # Number of CPU cores to use

DC_type = 'Dopen_nok'  # 'Dopen_nok', 'Dopen', 'D5_nok', 'D5'
KI_type = 'KI2'        # 'KI2', 'KI2_frac', 'KI2_adj', 'KI2_adj_frac'
file_path = '%s/data/processed/PaperID_%s-%s_merged.pickle'%(pre_path,KI_type,DC_type)

all_vars =[KI_type,DC_type,'Year','Field','log_C5','Multidiscipline','Novelty_90pct','reference_Count','reference_Age','reference_C5','reference_'+DC_type,'Team_Size','Team_Distance','Team_Freshness']
numeric_cols = ['Year','log_C5','Multidiscipline','Novelty_90pct','reference_Count','reference_Age','reference_C5','reference_DC'+DC_type,'Team_Size']
non_numeric_cols = ['Field'] # Categorical columns to be one-hot encoded
category_cols = ['Team_Distance','Team_Freshness']


# Define the regression type and treatment effect type
data_type = 'raw'       # data_type: raw or normalized
treat_effect_type='ATT' # treat_effect_type: ATT or ATC

# Load the data

df = load_data(file_path, data_type, all_vars, non_numeric_cols)

bin_rank_list = [0,10,20,30,40,50,60,70,80,90,100]
label_rank_list = [5,15,25,35,45,55,65,75,85,95,105]
# binning the rank of 'KI_type' and 'DC_type'
df[DC_type] = rank(df[DC_type].tolist())
df[KI_type] = rank_bin(df[KI_type].tolist(),bin_rank_list,label_rank_list)
# convert 'KI_type' to numeric
df[KI_type] = pd.to_numeric(df[KI_type])


# --- Define Covariates, Outcome, Treatment ---
Treatment = all_vars[0] # KI_type
Outcome = all_vars[1] # DC_type
Covariates = all_vars[2:] # All covariates except Treatment and Outcome

# Ensure no NaNs in relevant columns for original df before starting bootstrap
df = df.dropna(subset=Covariates + [Outcome, Treatment])
print(df)

treatment_size_list = sorted(list(df[Treatment].drop_duplicates()))
print(f"Treatment sizes: {treatment_size_list}")


print(f"--- Running PSM analysis with {n_bootstraps} bootstraps ---")
for control_size in treatment_size_list:

    print('The baseline:',control_size)

    data_effect_size = []

    for treatment_size in treatment_size_list:
        if treatment_size == control_size:  # if the treatment size is equal to the control size, skip
            data_effect_size.append([treatment_size, 0, treat_effect_type, 1, 0, 0]) # Effect Size=0, P value=1, CI lower=0, CI upper=0
        else:
            print(f'Treatment Size: {treatment_size}')
            # filter the dataframe for the current treatment and control sizes
            df_temp = df[df[Treatment].isin([control_size, treatment_size])]
                        
            overall_att, p_value, ci_lower, ci_upper = PSM_analysis_bootstrap(
                df_temp, Treatment, Outcome, non_numeric_cols, control_size, 
                treatment_size, treat_effect_type, n_bootstraps, CPU_COUNT)
            data_effect_size.append([treatment_size, overall_att, treat_effect_type, p_value, ci_lower, ci_upper])
    
    effect_size = pd.DataFrame(data_effect_size, columns=['Treatment Size', 'Effect Size','Effect Type', 
                                                        'P value', 'CI lower', 'CI upper'])
    effect_size.to_csv('%s/results/results_for_tables/PSM_Analysis_Results/ALL_%s_Controlled_%s.csv'%(pre_path,treat_effect_type,control_size),encoding='utf-8',index=False)

## 5. Coarsened Exact Matching (CEM) analysis
To further enhance the robustness of our findings by Propensity Score Matching (PSM), we have performed an additional causal inference analysis using Coarsened Exact Matching (CEM). CEM is a non-parametric matching method that directly coarsens the covariates into bins and then performs exact matching within these coarsened bins. This ensures excellent covariate balance within the matched samples by guaranteeing that all matched units have identical values on the coarsened covariates, thereby reducing model dependence and confounding bias more effectively than traditional PSM in some scenarios.

In [ ]:
from sklearn.linear_model import LinearRegression
from concurrent.futures import ProcessPoolExecutor, as_completed


def create_coarsening_schema_optimized(numeric_cols, df_sample, bin_number):
    """
    create coarsening schema - support different bin numbers

    Parameters:
    -----------
    numeric_cols : list

    df_sample : pd.DataFrame

    bin_number : int
    
    Returns:
    --------
    dict : coarsening_schema
    """
    coarsening_schema = {}
    
    for var in numeric_cols:
        if var not in df_sample.columns:
            continue
            
        if var == 'Year':
            # static bins for Year
            coarsening_schema[var] = (pd.cut, {'bins': bin_number, 'duplicates': 'drop'})
        elif var == 'Team_Size':
            # static bins for Team_Size
            if bin_number == 3:
                bins = [0, 3, 6, np.inf]
            elif bin_number == 4:
                bins = [0, 2, 4, 8, np.inf]
            elif bin_number == 5:
                bins = [0, 2, 4, 6, 8, np.inf]
            coarsening_schema[var] = (pd.cut, {'bins': bins, 'duplicates': 'drop'})
        else:
            # other variables use quantile binning
            unique_vals = df_sample[var].nunique()
            if unique_vals < bin_number:
                continue

            # calculate quantiles as fixed cut points
            quantiles = np.linspace(0, 1, bin_number + 1)
            quantile_values = np.quantile(df_sample[var].dropna(), quantiles)

            # duplicate to keep at least k bins
            quantile_values = np.unique(quantile_values)
            if len(quantile_values) >= 3:
                coarsening_schema[var] = (pd.cut, {
                    'bins': quantile_values, 
                    'duplicates': 'drop', 
                    'include_lowest': True
                })
            else:
                coarsening_schema[var] = (pd.cut, {'bins': bin_number, 'duplicates': 'drop'})
    
    return coarsening_schema


def coarsen_variables_optimized(df, Treatment, coarsening_schema):
    """
    quick variable coarsening
    """
    df_coarsened = df.copy()

    for var, (func, kwargs) in coarsening_schema.items():
        if var == Treatment or var not in df.columns:
            continue
        try:
            df_coarsened[var] = func(df[var], **kwargs)
        except Exception as e:
            continue
    
    return df_coarsened



def preprocess_features_once_CEM(df_temp, Treatment, Outcome, treatment_size, non_numeric_cols, bin_number):
    """
    combine structured arrays, including features_coarse
    """

    df_encoded = df_temp.copy()
    df_encoded[Treatment] = (df_encoded[Treatment] == treatment_size).astype(int)

    # 1. One-hot encoding
    if non_numeric_cols:
        df_encoded = pd.get_dummies(df_encoded, columns=non_numeric_cols, drop_first=True, dtype=int)

    feature_cols = [col for col in df_encoded.columns if col not in [Treatment, Outcome]]

    # 2. optimize data types
    X_features = df_encoded[feature_cols+[Treatment]].values#.astype(np.float32)  # with Treatment, for weighted linear regression
    y_treatment = df_encoded[Treatment].values
    y_outcome = df_encoded[Outcome].values.astype(np.float32)

    # 3. prepare coarsening
    coarsening_schema = create_coarsening_schema_optimized(numeric_cols, df_encoded, bin_number)
    df_coarsened = coarsen_variables_optimized(df_encoded, Treatment, coarsening_schema)
    X_coarsened = df_coarsened[feature_cols].values  # without Treatment, just for coarsening

    # 4. combine structured arrays, including features_coarse
    n_samples, n_features = X_coarsened.shape
    
    dtype_list = [
        ('features', X_features.dtype, (n_features+1,)),        # raw features, including Treatment
        ('features_coarse', X_coarsened.dtype, (n_features,)),  # coarse features, excluding Treatment
        ('treatment', y_treatment.dtype), 
        ('outcome', np.float32)
    ]
    
    structured_array = np.empty(n_samples, dtype=dtype_list)
    structured_array['features'] = X_features
    structured_array['features_coarse'] = X_coarsened
    structured_array['treatment'] = y_treatment
    structured_array['outcome'] = y_outcome
    
    preprocessing_info = {
        'structured_data': structured_array,
        'n_features': n_features+1,  # including Treatment
        'feature_names': feature_cols+[Treatment],
        'n_samples': n_samples,
        'treatment_name': Treatment
    }
    
    return preprocessing_info


def create_strata_numpy(X_coarse, treatment_mask, control_mask):
    """
    使用numpy操作的快速分层创建
    """
    try:
        n_samples, n_features = X_coarse.shape

        # use hash method to create strata
        strata_dict = {}
        
        for i in range(n_samples):
            # create tuple key (faster hash)
            key = tuple(X_coarse[i])
            
            if key not in strata_dict:
                strata_dict[key] = {'treatment': [], 'control': []}
            
            if treatment_mask[i]:
                strata_dict[key]['treatment'].append(i)
            elif control_mask[i]:
                strata_dict[key]['control'].append(i)

        # vectorized filtering
        valid_strata = {k: v for k, v in strata_dict.items()
                       if len(v['treatment']) > 0 and len(v['control']) > 0}
        
        return valid_strata
        
    except Exception:
        return {}


def calculate_CEM_weights_numpy(strata_dict, n_samples):
    """
    use numpy for fast weight calculation
    """
    weights = np.zeros(n_samples, dtype=np.float32)

    # batch process weight assignment
    for strata_data in strata_dict.values():
        treatment_indices = np.array(strata_data['treatment'], dtype=np.int32)
        control_indices = np.array(strata_data['control'], dtype=np.int32)
        
        if len(treatment_indices) > 0:
            weights[treatment_indices] = 1.0 / len(treatment_indices)
        
        if len(control_indices) > 0:
            weights[control_indices] = 1.0 / len(control_indices)

    # vectorized normalization
    weights_sum = np.sum(weights)
    if weights_sum > 0:
        weights *= n_samples / weights_sum
    
    return weights


def CEM_batch_process(args):
    """batch processing"""
    preprocessing_info, control_size, treatment_size, treat_effect_type, seed_batch = args
    
    try:
        results = []
        
        # obtain all necessary data at once
        n_samples = preprocessing_info['n_samples']
        feature_names = preprocessing_info['feature_names']
        treatment_name = preprocessing_info['treatment_name']
        
        # pre-compute treatment column index
        treatment_col_idx = feature_names.index(treatment_name)
        
        for seed in seed_batch:
            try:
                # 1. generate bootstrap indices
                np.random.seed(seed)
                bootstrap_indices = np.random.randint(0, n_samples, size=n_samples)

                # 2. direct indexing to avoid redundant data extraction
                bootstrap_data = preprocessing_info['structured_data'][bootstrap_indices]
                
                X_bootstrap = bootstrap_data['features']
                X_coarse_bootstrap = bootstrap_data['features_coarse']
                y_treatment_bootstrap = bootstrap_data['treatment']
                y_outcome_bootstrap = bootstrap_data['outcome']

                # 3. vectorization
                treatment_binary = (y_treatment_bootstrap == 1)
                control_binary = (y_treatment_bootstrap == 0)

                # quick check
                if not (treatment_binary.any() and control_binary.any()):
                    continue

                # 4. quick strata creation
                strata_dict = create_strata_numpy(X_coarse_bootstrap, treatment_binary, control_binary)
                
                if len(strata_dict) < 2:
                    continue

                # 5. quick weight calculation
                weights = calculate_CEM_weights_numpy(strata_dict, n_samples)

                # 6. check weight validity
                valid_mask = weights > 1e-8
                if valid_mask.sum() < 4:
                    continue

                # 7. direct regression to avoid redundant model creation
                X_weighted = X_bootstrap[valid_mask]
                y_weighted = y_outcome_bootstrap[valid_mask]
                weights_valid = weights[valid_mask]


                model = LinearRegression()
                model.fit(X_weighted, y_weighted, sample_weight=weights_valid)
                
                if len(model.coef_) > treatment_col_idx:
                    att = float(model.coef_[treatment_col_idx])
                    if np.isfinite(att):
                        results.append(att)
                    else:
                        continue

            except Exception:
                continue
        
        return results
        
    except Exception:
        return []


def CEM_analysis_bootstrap(df_temp, Treatment, Outcome, non_numeric_cols, control_size, treatment_size, treat_effect_type, n_bootstraps, CPU_COUNT, bin_number):

    # 1. preprocessing
    preprocessing_info = preprocess_features_once_CEM(df_temp, Treatment, Outcome, treatment_size, non_numeric_cols, bin_number)

    # 2. batch strategy
    batch_size = max(1, n_bootstraps // (CPU_COUNT * 2))
    seed_batches = []
    
    for i in range(0, n_bootstraps, batch_size):
        batch_seeds = list(range(42 + i, 42 + min(i + batch_size, n_bootstraps)))
        seed_batches.append(batch_seeds)
    
    args_list = [(preprocessing_info, control_size, treatment_size, treat_effect_type, batch_seeds) 
                 for batch_seeds in seed_batches]
    
    att_bootstraps = []

    # set less process but increase each process workload
    optimal_workers = min(CPU_COUNT, len(seed_batches))

    # 3. parallel processing
    with ProcessPoolExecutor(max_workers=optimal_workers) as executor:
        futures = [executor.submit(CEM_batch_process, args) for args in args_list]
        
        for i, future in enumerate(as_completed(futures)):
            if i % max(1, len(futures) // 10) == 0:
                print(f"    CEM: Completed batch {i+1}/{len(futures)}")
            
            batch_results = future.result()
            att_bootstraps.extend(batch_results)

    # 4. statistical calculation
    if not att_bootstraps:
        # print("    CEM: No valid bootstrap samples found.")
        return np.nan, np.nan, np.nan, np.nan
    
    att_array = np.array(att_bootstraps, dtype=np.float32)
    
    if len(att_array) < 2:
        # print("    CEM: Not enough valid bootstrap samples for statistical analysis.")
        return float(np.mean(att_array)), np.nan, np.nan, np.nan

    # Vectorized statistical calculation
    overall_att = float(np.mean(att_array))
    ci_lower = float(np.percentile(att_array, 2.5))
    ci_upper = float(np.percentile(att_array, 97.5))

    # Fast p-value calculation
    positive_prop = np.mean(att_array > 0)
    p_value = float(2 * min(positive_prop, 1 - positive_prop))
    
    print(f"    CEM: Overall ATT: {overall_att:.4f}, P-value: {p_value:.4f}, CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
    return overall_att, p_value, ci_lower, ci_upper

In [ ]:
# Set parameters
n_bootstraps = 1000      # Number of iterations for bootstrapping
CPU_COUNT = 36           # Number of CPU cores to use
bin_number = 3           # Number of bins for coarsening, [3,4,5]

DC_type = 'Dopen_nok'  # 'Dopen_nok', 'Dopen', 'D5_nok', 'D5'
KI_type = 'KI2'        # 'KI2', 'KI2_frac', 'KI2_adj', 'KI2_adj_frac'
file_path = '%s/data/processed/PaperID_%s-%s_merged.pickle'%(pre_path,KI_type,DC_type)

all_vars =[KI_type,DC_type,'Year','Field','log_C5','Multidiscipline','Novelty_90pct','reference_Count','reference_Age','reference_C5','reference_'+DC_type,'Team_Size','Team_Distance','Team_Freshness']
numeric_cols = ['Year','log_C5','Multidiscipline','Novelty_90pct','reference_Count','reference_Age','reference_C5','reference_DC'+DC_type,'Team_Size']
non_numeric_cols = ['Field'] # Categorical columns to be one-hot encoded
category_cols = ['Team_Distance','Team_Freshness']


# Define the regression type and treatment effect type
data_type = 'raw'       # data_type: raw or normalized
treat_effect_type='ATT' # treat_effect_type: ATT or ATC

# Load the data
df = load_data(file_path, data_type, all_vars, non_numeric_cols)

bin_rank_list = [0,10,20,30,40,50,60,70,80,90,100]
label_rank_list = [5,15,25,35,45,55,65,75,85,95,105]
# binning the rank of 'KI_type' and 'DC_type'
df[DC_type] = rank(df[DC_type].tolist())
df[KI_type] = rank_bin(df[KI_type].tolist(),bin_rank_list,label_rank_list)
# convert 'KI_type' to numeric
df[KI_type] = pd.to_numeric(df[KI_type])


# --- Define Covariates, Outcome, Treatment ---
Treatment = all_vars[0] # DR_tag
Outcome = all_vars[1] # DC_tag
Covariates = all_vars[2:] # All covariates except Treatment and Outcome

# Ensure no NaNs in relevant columns for original df before starting bootstrap
df = df.dropna(subset=Covariates + [Outcome, Treatment])
print(df)

treatment_size_list = sorted(list(df[Treatment].drop_duplicates()))
print(f"Treatment sizes: {treatment_size_list}")

print(f"--- Running CEM analysis with {n_bootstraps} bootstraps ---")
for control_size in treatment_size_list:

    print('The baseline:',control_size)

    data_effect_size = []

    control_mask = (df[Treatment] == control_size)

    for treatment_size in treatment_size_list:
        if treatment_size == control_size:  # if the treatment size is equal to the control size, skip
            data_effect_size.append([treatment_size, 0, treat_effect_type, 1, 0, 0]) # Effect Size=0, P value=1, CI lower=0, CI upper=0
        else:
            print(f'Treatment Size: {treatment_size}')
            # filter the dataframe for the current treatment and control sizes
            df_temp = df[df[Treatment].isin([control_size, treatment_size])]
                        
            overall_att, p_value, ci_lower, ci_upper = CEM_analysis_bootstrap(
                df_temp, Treatment, Outcome, non_numeric_cols, control_size, 
                treatment_size, treat_effect_type, n_bootstraps, CPU_COUNT, bin_number)
            data_effect_size.append([treatment_size, overall_att, treat_effect_type, p_value, ci_lower, ci_upper])
    
    effect_size = pd.DataFrame(data_effect_size, columns=['Treatment Size', 'Effect Size','Effect Type', 
                                                        'P value', 'CI lower', 'CI upper'])
    effect_size.to_csv('%s/results/results_for_tables/CEM_Analysis_Results/ALL_%s_Controlled_%s.csv'%(pre_path,treat_effect_type,control_size),encoding='utf-8',index=False)